In [1]:
from pathlib import Path
p = Path("../src/evaluate.py")
s = p.read_text()

if "_pr_auc" not in s:
    s = s.replace('''def score_row(y, p, threshold: float = 0.5) -> dict:''', '''def _pr_auc(y, p) -> float:
    """PR-AUC with ties broken by tiny seeded noise.

    A model with few distinct scores (e.g. a shipping-mode lookup) has huge tied
    blocks, and sklearn scores a tied block as one lump, understating PR-AUC.
    Breaking ties the same way for every model keeps comparisons fair.
    """
    rng = np.random.default_rng(config.RANDOM_STATE)
    return average_precision_score(y, np.asarray(p) + rng.normal(0, 1e-7, len(p)))


def score_row(y, p, threshold: float = 0.5) -> dict:''')
    s = s.replace('"pr_auc": average_precision_score(y, p),', '"pr_auc": _pr_auc(y, p),')
    p.write_text(s)
print("patched:", "_pr_auc" in p.read_text())

patched: True


In [2]:
%%writefile ../src/models.py
"""Model factories. Every model is a full Pipeline: preprocessing + estimator."""
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from src import config
from src.features import build_preprocessor

# Feature groups, used to ask "does this group add anything beyond shipping mode?"
FEATURE_GROUPS = {
    "geography": ["Market", "Order Region", "Order Country", "Customer State"],
    "customer": ["Customer Segment", "Type", "Customer Country"],
    "product": ["Category Name", "Department Name", "Product Name"],
    "order_value": ["Order Item Discount", "Order Item Discount Rate",
                    "Order Item Product Price", "Order Item Quantity",
                    "Order Item Total"],
    "coordinates": ["Latitude", "Longitude"],
}


def _split(cols, numeric):
    num = [c for c in cols if c in numeric]
    cat = [c for c in cols if c not in numeric]
    return num, cat


def make_logreg(cols, numeric, C: float = 1.0) -> Pipeline:
    num, cat = _split(cols, numeric)
    return Pipeline([("prep", build_preprocessor(num, cat)),
                     ("clf", LogisticRegression(C=C, max_iter=2000))])


def make_hgb(cols, numeric, **params) -> Pipeline:
    """Gradient boosting with integer-coded categories (unseen -> missing)."""
    num, cat = _split(cols, numeric)
    prep = ColumnTransformer(
        [("num", "passthrough", num),
         ("cat", OrdinalEncoder(handle_unknown="use_encoded_value",
                                unknown_value=np.nan,
                                encoded_missing_value=np.nan), cat)],
        verbose_feature_names_out=False)
    mask = [False] * len(num) + [True] * len(cat)
    clf = HistGradientBoostingClassifier(categorical_features=mask,
                                         random_state=config.RANDOM_STATE, **params)
    return Pipeline([("prep", prep), ("clf", clf)])

Writing ../src/models.py


In [3]:
import sys
sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from src import config
from src.features import split_feature_types
from src.models import make_logreg, make_hgb, FEATURE_GROUPS
from src.evaluate import fold_scores

train = pd.read_parquet(config.ROOT / "data/processed/train.parquet")
y, groups = train[config.TARGET], train["Customer Id"]
numeric, categorical = split_feature_types(train)
all_feats = numeric + categorical
MODE = [config.SHIP_MODE]

candidates = {
    "mode only (LR)":         (make_logreg(MODE, numeric), MODE),
    "LR, all features":       (make_logreg(all_feats, numeric, C=1.0), all_feats),
    "LR, all, strong reg.":   (make_logreg(all_feats, numeric, C=0.01), all_feats),
    "HGB, all features":      (make_hgb(all_feats, numeric, max_depth=3, learning_rate=0.05,
                                        max_iter=200, early_stopping=False), all_feats),
}
for group, cols in FEATURE_GROUPS.items():
    cc = MODE + cols
    candidates[f"mode + {group}"] = (make_logreg(cc, numeric), cc)

print(len(candidates), "candidates")

9 candidates


In [4]:
fold_results = {}
for name, (model, cols) in candidates.items():
    fold_results[name] = fold_scores(model, train[cols], y, groups)
    print("done:", name)

base = fold_results["mode only (LR)"]
rows = []
for name, f in fold_results.items():
    rows.append({
        "model": name,
        "roc_auc": f["roc_auc"].mean(), "auc_std": f["roc_auc"].std(),
        "pr_auc": f["pr_auc"].mean(),
        "log_loss": f["log_loss"].mean(), "brier": f["brier"].mean(),
        "d_auc_vs_mode": (f["roc_auc"] - base["roc_auc"]).mean(),
        "d_logloss_vs_mode": (f["log_loss"] - base["log_loss"]).mean(),
        "folds_auc_better": int((f["roc_auc"] > base["roc_auc"]).sum()),
    })
comparison = pd.DataFrame(rows).set_index("model").round(4)
comparison

done: mode only (LR)
done: LR, all features
done: LR, all, strong reg.
done: HGB, all features
done: mode + geography
done: mode + customer
done: mode + product
done: mode + order_value
done: mode + coordinates


,roc_auc,auc_std,pr_auc,log_loss,brier,d_auc_vs_mode,d_logloss_vs_mode,folds_auc_better
model,,,,,,,,
mode only (LR),0.7411,0.0049,0.8343,0.5380,0.1884,0.0000,0.0000,0
"LR, all features",0.7390,0.0054,0.8327,0.5406,0.1896,-0.0021,0.0027,1
"LR, all, strong reg.",0.7387,0.0050,0.8325,0.5417,0.1890,-0.0024,0.0037,1
"HGB, all features",0.7393,0.0060,0.8321,0.5421,0.1901,-0.0018,0.0041,0
mode + geography,0.7395,0.0052,0.8329,0.5400,0.1893,-0.0016,0.0020,1
mode + customer,0.7380,0.0040,0.8326,0.5382,0.1885,-0.0031,0.0002,0
mode + product,0.7409,0.0051,0.8343,0.5382,0.1885,-0.0002,0.0002,3
mode + order_value,0.7407,0.0045,0.8343,0.5380,0.1884,-0.0004,0.0000,1
mode + coordinates,0.7397,0.0056,0.8337,0.5380,0.1884,-0.0014,0.0000,1
